[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/drigaa/rEVA_CE4IB/HEAD?urlpath=%2Fdoc%2Ftree%2Fcasestudy.ipynb)

![thermal-network](images/Screenshot%202026-10-29%20093356.png)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from dm4bem import read_epw, sol_rad_tilt_surf

In [ ]:
filename = Path("weather_data/Hombrechtikon/CHE_ZH_Hombrechtikom.068510_TMYx.2011-2025.epw")
data, meta = read_epw(filename, coerce_year=None)

## Connect this to data
T_0 = 10

In [ ]:
data.value_counts(subset=["year", "month"])

In [ ]:
weather_data = data[["temp_air", "dir_n_rad", "dif_h_rad"]]

In [ ]:
# radiative properties
ε_wLW = 0.85    # long wave emmisivity: wall surface (concrete)
ε_gLW = 0.90    # long wave emmisivity: glass pyrex
α_wSW = 0.25    # short wave absortivity: white smooth surface
α_gSW = 0.38    # short wave absortivity: reflective blue glass
τ_gSW = 0.30    # short wave transmitance: reflective blue glass

sbc = 5.67e-8

In [ ]:
air = {
    'Density': 1.2,
    'Specific heat': 1000
}

In [ ]:
l = 3

concrete = {
    'Conductivity': 1.400,
    'Density': 2300.0,
    'Specific heat': 880,
    'Width': 0.15,
    'Surface': l**2
}

insulation = {
    'Conductivity': 0.040,
    'Density': 16.0,
    'Specific heat': 1210,
    'Width': 0.25,
    'Surface': l**2
}

glass = {
    'U-Value': 0.6,
    'Surface': l**2
}

wall = pd.DataFrame.from_dict(
    {
        'Layer_out': insulation,
        'Layer_in': concrete,
        'window': glass
    },
    orient='index')

wall

In [ ]:
# convection coefficients, W/(m²·K)
h = pd.DataFrame([{'in': 4., 'out': 10.}], index=['h'])
h

In [ ]:
## P-controler gain
Kp = 0
Ti_sp = 30

In [ ]:
number_of_nodes = 10
number_of_branches = 14
G_diag = np.zeros(number_of_branches)

G_diag[[1, 2]] = (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_out
G_diag[[3, 4, 7 ,8]] =  (wall["Conductivity"] / (wall["Width"] / 2) * wall["Surface"]).Layer_in

G_diag[[0]] = h['out'] * wall["Surface"].Layer_out
G_diag[[5, 6, 9]] = h['in'] * wall["Surface"].Layer_in

G_diag[[12, 13]] = (wall['U-Value'] * wall["Surface"]).window

G_diag[[10, 11]] = Kp

G = np.diag(G_diag)

In [ ]:
C_diag = np.zeros(number_of_nodes)

In [ ]:
C_diag[1] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_out
C_diag[[3, 7]] = (wall['Width'] * wall['Surface'] * wall['Density'] * wall["Specific heat"]).Layer_in
C_diag[[5, 9]] = air["Density"] * l**3 * air["Specific heat"]
C = np.diag(C_diag)
C

In [ ]:
A = np.zeros((14, 10))

In [ ]:
A[0, 0] = 1
A[1, 0], A[1, 1] = -1, 1
A[2, 1], A[2, 2] = -1, 1
A[3, 2], A[3, 3] = -1, 1
A[4, 3], A[4, 4] = -1, 1
A[5, 4], A[5, 5] = -1, 1
A[6, 6], A[6, 5] = -1, 1
A[7, 7], A[7, 6] = -1, 1
A[8, 8], A[8, 7] = -1, 1
A[9, 9], A[9, 8] = -1, 1
A[10, 9] = 1
A[11, 5] = 1
A[12, 9] = 1
A[13, 5] = 1

In [ ]:
b = np.zeros(number_of_branches)
b[[0, 12, 13]] = T_0
b[[10, 11]] = Ti_sp

In [ ]:
f = np.zeros(number_of_nodes)

In [ ]:
temp_steady_state = np.linalg.inv(A.T @ G @ A) @ (A.T @ G @ b + f)
pd.Series(temp_steady_state)